In [1]:
"""
make_figure3_final.py
=====================
Generates Figure 3 — Social Infrastructure vs. Log Output per Worker,
Extension (2019), Sample B, N = 111 — in the exact same style as Figure 2.

Reads:  extension_clean.csv  (place in same folder as this script, or update DATA_PATH)
Writes: scatter_3_final.png

Style matches Figure 2 exactly:
  - Two-line bold centred title
  - Stat box top-left with navy border
  - Teal OLS fit line + teal 95% CI shaded band
  - Navy dots  = observed S  (N=99)
  - Gold dots  = imputed S   (N=12)
  - ISO-3 country labels in matching colour
  - Legend lower-right
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from scipy import stats

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_PATH = r"C:\Users\Adams\OneDrive\DE & E Research\outputs\sample_B_replication_extended.csv"
OUT_PATH  = r"C:\Users\Adams\OneDrive\DE & E Research\outputs\scatter_3_final.png"

# ── Colour palette (identical to Figure 2) ────────────────────────────────
NAVY       = '#1B2A4A'
GOLD       = '#C8901A'
TEAL       = '#1A6B72'
TEAL_LIGHT = '#D6E8EA'

# ── Figure 2 layout constants ──────────────────────────────────────────────
FIG_W, FIG_H = 13.5, 8.5
TITLE_FS     = 15
LABEL_FS     = 12
TICK_FS      = 10
ANNOT_FS     = 7.2
DOT_SIZE     = 32
DOT_ALPHA    = 0.85
LINE_W       = 2.2

# ── Load data ──────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH, sep='\t')
df['log_yl']     = np.log(df['yl_adj'])
df['s_imputed']  = df['si_was_imputed'].astype(bool)

n_obs = int((~df['s_imputed']).sum())
n_imp = int(df['s_imputed'].sum())
n_tot = len(df)
print(f"Sample B:  N={n_tot}  (observed={n_obs},  imputed={n_imp})")
print(f"Imputed: {list(df[df['s_imputed']]['iso3'])}")

# ── OLS regression ─────────────────────────────────────────────────────────
x = df['social_infra'].values
y = df['log_yl'].values
slope, intercept, r_value, p_value, _ = stats.linregress(x, y)
r2    = r_value ** 2
stars = "***" if p_value < 0.001 else ("**" if p_value < 0.01 else "*")
print(f"OLS:  slope={slope:.3f}{stars},  R²={r2:.3f},  N={n_tot}")

# ── 95% confidence band ────────────────────────────────────────────────────
x_line  = np.linspace(x.min(), x.max(), 300)
y_line  = intercept + slope * x_line
n, x_m  = len(x), x.mean()
resid2  = np.sum((y - (intercept + slope * x)) ** 2) / (n - 2)
se_band = np.sqrt(resid2 * (1/n + (x_line - x_m)**2 / np.sum((x - x_m)**2)))
t_crit  = stats.t.ppf(0.975, df=n - 2)
ci_lo   = y_line - t_crit * se_band
ci_hi   = y_line + t_crit * se_band

# ── Build figure ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# Grid
ax.grid(True, color='#CCCCCC', linewidth=0.5, linestyle='-', alpha=0.7, zorder=0)
ax.set_axisbelow(True)

# 95% CI shaded band
ax.fill_between(x_line, ci_lo, ci_hi, color=TEAL_LIGHT, alpha=0.55, zorder=1)

# Scatter: observed (navy) then imputed (gold) on top
mask_obs = ~df['s_imputed'].values
mask_imp =  df['s_imputed'].values
ax.scatter(x[mask_obs], y[mask_obs],
           color=NAVY, s=DOT_SIZE, alpha=DOT_ALPHA, linewidths=0, zorder=3)
ax.scatter(x[mask_imp], y[mask_imp],
           color=GOLD, s=DOT_SIZE, alpha=DOT_ALPHA, linewidths=0, zorder=4)

# OLS fit line
ax.plot(x_line, y_line, color=TEAL, linewidth=LINE_W, zorder=5)

# ── Country ISO labels ─────────────────────────────────────────────────────
LABEL_ISOS = {
    # High income
    'USA','CHE','NOR','JPN','GBR','FRA','DEU','CAN','AUS','IRL','NZL',
    'SGP','HKG','SWE','DNK','FIN','AUT','NLD','BEL','ISR','KOR','TWN',
    'ISL','LUX','MLT','CYP',
    # Upper-middle
    'BRA','MEX','ARG','COL','THA','MYS','TUR','ZAF','IRN','DZA',
    'PER','ECU','EGY','MAR','TUN','URY','CHL','POL','ROU','HUN',
    'BGR','CZE','SVK','CRI','PAN','DOM','GAB','CHN','BWA',
    # Lower-middle and low
    'IND','PAK','BGD','NGA','KEN','TZA','NER','GHA','SDN',
    'UGA','ZWE','MOZ','MLI','MWI','CAF','BDI','RWA','LAO',
    'HTI','YEM','VEN','SYR','IRQ','QAT','KWT','SAU','TWN',
    # Imputed (always label these)
    'ARE','BEN','BLZ','BRB','FJI','MRT','MUS','NPL',
}

for _, row in df.iterrows():
    iso = str(row['iso3'])
    if iso in LABEL_ISOS:
        col = GOLD if row['s_imputed'] else NAVY
        ax.annotate(
            iso,
            xy=(row['social_infra'], row['log_yl']),
            xytext=(3, 2),
            textcoords='offset points',
            fontsize=ANNOT_FS,
            color=col,
            zorder=6,
        )

# ── Stat box top-left (matching Figure 2 exactly) ─────────────────────────
ax.text(
    0.02, 0.97,
    f"R² = {r2:.3f}  |  Slope = {slope:.3f}{stars}  |  N = {n_tot}",
    transform=ax.transAxes,
    fontsize=11, fontweight='bold', color=NAVY,
    va='top', ha='left',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
              edgecolor=NAVY, linewidth=1.2, alpha=0.95),
    zorder=7,
)

# ── Two-line bold centred title (matching Figure 2) ────────────────────────
ax.set_title(
    "Social Infrastructure vs. Output per Worker  —  Sample B, Extension (2019)\n"
    "Navy = observed S  ·  Gold = imputed S  ·  Shaded region = 95% CI",
    fontsize=TITLE_FS, fontweight='bold', pad=12, color='black',
)

# ── Axis labels ────────────────────────────────────────────────────────────
ax.set_xlabel("Social Infrastructure  (S)", fontsize=LABEL_FS, labelpad=8)
ax.set_ylabel("log(Output per Worker)",      fontsize=LABEL_FS, labelpad=8)
ax.tick_params(axis='both', labelsize=TICK_FS, length=4)

# ── Legend lower-right (matching Figure 2) ────────────────────────────────
ax.legend(
    handles=[
        Line2D([0],[0], color=TEAL, linewidth=LINE_W,
               label=f'OLS fit  (slope={slope:.2f}, R²={r2:.3f})'),
        mpatches.Patch(color=NAVY, label=f'Observed S  (N={n_obs})'),
        mpatches.Patch(color=GOLD, label=f'Imputed S  (N={n_imp})'),
    ],
    loc='lower right',
    fontsize=9.5, framealpha=0.92, edgecolor='#CCCCCC', fancybox=False,
)

# ── Spine styling ──────────────────────────────────────────────────────────
for sp in ax.spines.values():
    sp.set_edgecolor('#AAAAAA')
    sp.set_linewidth(0.8)

# ── Save ───────────────────────────────────────────────────────────────────
plt.tight_layout()
fig.savefig(OUT_PATH, dpi=160, bbox_inches='tight', facecolor='white')
plt.close()
print(f"\nSaved  →  {OUT_PATH}")


KeyError: 'yl_adj'